In [1]:
import torch
import argparse
import random
import os
import numpy as np
import pandas as pd
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, GATConv
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
import pickle as pkl

/usr/local/lib/python3.8/dist-packages/torch_geometric/typing.py:72: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: libtorch_cuda_cpp.so: cannot open shared object file: No such file or directory
  warnings.warn(f"An issue occurred while importing 'torch-scatter'. "


In [4]:
# 设置随机种子
def set_seed(seed=7):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True


class GCN(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(GCN, self).__init__()
        self.conv1 = GCNConv(input_dim, hidden_dim, normalize=False)
        # self.conv2 = GCNConv(hidden_dim, hidden_dim, normalize=False)
        self.mlp = MLP(hidden_dim, output_dim)
        self.dropout = nn.Dropout()
    def forward(self, x, edge_index, edge_weight):
        x = self.conv1(x, edge_index, edge_weight)
        x = F.relu(x)
        x = self.dropout(x)
        # x = self.conv2(x, edge_index, edge_weight)
        # x = F.relu(x)
        # x = self.dropout(x)
        x = self.mlp(x)
        return x

class GAT(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(GAT, self).__init__()
        self.conv1 = GATConv(input_dim, hidden_dim, normalize=False)
        # self.conv2 = GATConv(hidden_dim, hidden_dim, normalize=False)
        self.mlp = MLP(hidden_dim, output_dim)
        self.dropout = nn.Dropout()
    def forward(self, x, edge_index, edge_weight):
        x = self.conv1(x, edge_index, edge_weight)
        x = F.relu(x)
        x = self.dropout(x)
        # x = self.conv2(x, edge_index, edge_weight)
        # x = F.relu(x)
        # x = self.dropout(x)
        x = self.mlp(x)
        return x

class Union_GNN(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(Union_GNN, self).__init__()
        self.conv_GAT = GATConv(input_dim, hidden_dim, normalize=False)
        self.conv_GCN = GCNConv(hidden_dim, hidden_dim, normalize=False)
        self.mlp = MLP(hidden_dim, output_dim)
        self.dropout = nn.Dropout()
    def forward(self, graph):
        semantic = self.conv_GAT(graph.x, graph.semantic_edge_index, graph.semantic_edge_weight)
        semantic = F.relu(semantic)
        semantic = self.dropout(semantic)

        citation =  self.conv_GCN(semantic, graph.newstrcture_edge_index, graph.newstrcture_edge_weight)
        citation = F.relu(citation)
        citation = self.dropout(citation)

        y_hat = self.mlp(citation)
        # y_hat = self.mlp(semantic+citation)
        # y_hat = self.mlp(torch.cat([semantic+citation]))  # self.mlp = MLP(hidden_dim * 2, output_dim)

        return y_hat

class MLP(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(MLP, self).__init__()
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, input_dim),
            nn.ReLU(),
            nn.Dropout(),
            nn.Linear(input_dim, output_dim))
    def forward(self, x):
        return self.mlp(x)

In [3]:
i = 2019
distance_matrix = pd.read_csv(f'datas\prepare_finalmatrix_data\\embedding_distance_{i-2}_{i}.csv').values[:, 1:]
newstrcture_matrix = pd.read_csv(f'datas\prepare_finalmatrix_data\\keyword_cite_matrix_{i-2}_{i}-{i+1}.csv').values[:, 1:]
hotness = pd.read_csv(f'datas\prepare_finalmatrix_data\\kewords_count_matrix_{i-9}-{i}_filter_{i+1}.csv').values[:, 1:]

In [5]:
distance_matrix

array([[1.0, -0.3269027713596311, 0.6084901022090562, ...,
        -0.7991761729957602, -0.7986755022067711, -0.7507277764198532],
       [-0.3269027713596311, 0.9999999999999996, -0.4967816001844648,
        ..., 0.3398693689921214, 0.17878071028480033,
        0.38682310702773576],
       [0.6084901022090562, -0.4967816001844648, 1.0000000000000002, ...,
        -0.5317614835829327, -0.5494168037387984, -0.5228021657449544],
       ...,
       [-0.7991761729957602, 0.3398693689921214, -0.5317614835829327,
        ..., 1.0000000000000002, 0.9095285801634768, 0.9274010482566106],
       [-0.7986755022067711, 0.17878071028480033, -0.5494168037387984,
        ..., 0.9095285801634768, 1.0000000000000002, 0.8789466413744659],
       [-0.7507277764198532, 0.38682310702773576, -0.5228021657449544,
        ..., 0.9274010482566106, 0.8789466413744659, 1.0]], dtype=object)

In [8]:
# graph_list = pkl.load(open('graph_data.pkl', 'rb'))

graph_list = []
for i in range(2019, 2023):
    distance_matrix = pd.read_csv(f'datas\prepare_finalmatrix_data\\embedding_distance_{i-2}_{i}.csv').values[:, 1:]
    newstrcture_matrix = pd.read_csv(f'datas\prepare_finalmatrix_data\\keyword_cite_matrix_{i-2}_{i}-{i+1}.csv').values[:, 1:]
    hotness = pd.read_csv(f'datas\prepare_finalmatrix_data\\kewords_count_matrix_{i-9}-{i}_filter_{i+1}.csv').values[:, 1:]

    graph = Data()

    semantic_edge_weights = []
    semantic_edge_index = []
    num_nodes = distance_matrix.shape[0]
    print(num_nodes)
    for i in tqdm(range(num_nodes)):
        for j in range(i + 1, num_nodes):
            semantic_edge_weights.append(distance_matrix[i][j])
            semantic_edge_index.append([i, j])

    newstrcture_edge_weight = []
    newstrcture_edge_index = []
    num_nodes2 = newstrcture_matrix.shape[0]
    print(num_nodes2)
    for i in tqdm(range(num_nodes2)):
        for j in range(i + 1, num_nodes2):
            if newstrcture_matrix[i][j] != 0:
                newstrcture_edge_weight.append(newstrcture_matrix[i][j])
                newstrcture_edge_index.append([i, j])

    graph.semantic_edge_index = torch.tensor(semantic_edge_index).long().t().contiguous()
    graph.semantic_edge_weight = torch.tensor(semantic_edge_weights).float()

    graph.newstrcture_edge_index = torch.tensor(newstrcture_edge_index).long().t().contiguous()
    graph.newstrcture_edge_weight = torch.tensor(newstrcture_edge_weight).float()

    graph.x = torch.tensor(hotness[:, 0: -1].tolist()).float()
    graph.y = torch.tensor(hotness[:, -1].tolist()).float()

    print(len(semantic_edge_index))
    print(len(semantic_edge_weights))
    print(len(newstrcture_edge_weight))

    graph_list.append(graph)

with open(f'datas\prepare_finalmatrix_data\\graph_data.pkl','wb') as f:
    pkl.dump(graph_list,f)

1950


100%|██████████| 1950/1950 [00:02<00:00, 823.89it/s] 


1950


100%|██████████| 1950/1950 [00:00<00:00, 3176.43it/s]


1900275
1900275
35718
2094


...

2997576
2997576
57449
2636


100%|██████████| 2636/2636 [00:03<00:00, 720.65it/s] 


2636


100%|██████████| 2636/2636 [00:01<00:00, 2416.38it/s]


3472930
3472930
65652
